# Assignment 2: POTUS

---

## Task 1) President of the United States (Trump vs. Obama)

Surely, you're aware that the 45th President of the United States (@POTUS45) was an active user of Twitter, until (permanently) banned on Jan 8, 2021.
You can still enjoy his greatness at the [Trump Twitter Archive](https://www.thetrumparchive.com/). We will be using original tweets only, so make sure to remove all retweets.
Another fan of Twitter was Barack Obama (@POTUS43 and @POTUS44), who used the platform in a rather professional way.
Please also consider the POTUS Tweets of Joe Biden; we will be using those for testing.

### Data

There are multiple ways to get the data, but the easiest way is to download the files from the `Supplemental Materials` in the `Files` section of our Microsoft Teams group. 
Another way is to directly use the data from [Trump Twitter Archive](https://www.thetrumparchive.com/), [Obama Kaggle](https://www.kaggle.com/jayrav13/obama-white-house), and [Biden Kaggle](https://www.kaggle.com/rohanrao/joe-biden-tweets).
Before you get started, please download the files; you can put them into the data folder.

### N-gram Models

In this assignment, you will be doing some Twitter-related preprocessing and training n-gram models to be able to distinguish between Tweets of Trump, Obama, and Biden.
We will be using [NLTK](https://www.nltk.org), more specifically it's [`lm`](https://www.nltk.org/api/nltk.lm.html) module. 
Install the NLTK package within your working environment.
You can use some of the NLTK functions, but you have to implement the functions for likelihoods and perplexity from scratch.

*In this Jupyter Notebook, we will provide the steps to solve this task and give hints via functions & comments. However, code modifications (e.g., function naming, arguments) and implementation of additional helper functions & classes are allowed. The code aims to help you get started.*

---

In [33]:
# Dependencies
import csv
import json
import math
import random
import numpy as np
from nltk.tokenize import TweetTokenizer
from typing import List, Dict, Tuple, TypedDict, NamedTuple

### Prepare the Data

1.1 Prepare all the Tweets. Since the `lm` modules will work on tokenized data, implement a tokenization method that strips unnecessary tokens but retains special words such as mentions (@...) and hashtags (#...).

1.2 Partition into training and test sets; select about 100 tweets each, which we will be testing on later. As with any Machine Learning task, training and test must not overlap.

In [34]:
# Notice: ignore retweets 

class TrumpTweet(TypedDict):
    """The structure of the tweets in the JSON file"""
    id: int
    text: str
    isRetweet: str
    isDeleted: str
    device: str
    favorites: int
    retweets: int
    date: str
    isFlagged: str


def load_trump_tweets(filepath: str) -> List[str]:
    """Loads all Trump tweets and returns them as a list."""
    tweets: List[TrumpTweet]
    with open(filepath) as json_file:
        tweets = json.load(json_file)

    return [tweet["text"] for tweet in tweets if tweet["isRetweet"] == "f"]


ObamaTweet = TypedDict('ObamaTweet', {
    'Date': str,
    'Username': str,
    'Tweet-text': str,
    'Tweet Link': str,
    'Retweets': str,
    'Likes': str,
    'TweetImageUrl': str,
    'Image': str,
})


def load_obama_tweets(filepath: str) -> List[str]:
    """Loads all Obama tweets and returns them as a list."""
    tweets: List[ObamaTweet] = []
    with open(filepath) as csv_file:
        for row in csv.DictReader(csv_file):
            tweets.append(row)  # type: ignore

    return [tweet["Tweet-text"] for tweet in tweets]
    

class BidenTweet(TypedDict):
    """The structure of the tweets in the CSV file"""
    id: str
    timestamp: str
    url: str
    tweet: str
    replies: str
    retweets: str
    quotes: str
    likes: str


def load_biden_tweets(filepath: str) -> List[str]:
    """Loads all Biden tweets and returns them as a list."""
    tweets: List[BidenTweet] = []
    with open(filepath) as csv_file:
        for row in csv.DictReader(csv_file):
            tweets.append(row)  # type: ignore

    return [tweet["tweet"] for tweet in tweets]

In [35]:
# Notice: think about start and end tokens

NUM_TEST = 100


TokenList = List[str]
DataSet = List[TokenList]


class DataSplit(NamedTuple):
    """A data split into training and test"""
    train: DataSet
    test: DataSet


# we use the tweet tokenizer of NLTK
tokenizer = TweetTokenizer()


def tokenize(text) -> List[str]:
    """Tokenizes a single Tweet."""
    # add special tokens <s> and </s> to indicate begin and end of tweet
    return ["<s>"] + tokenizer.tokenize(text) + ["</s>"]
    

def split_and_tokenize(data: List[str], num_test:int=NUM_TEST, random_seed:int=42) -> DataSplit:
    """Splits and tokenizes the given list of Twitter tweets."""
    tokenized_data: DataSet = list(map(tokenize, data))

    random.seed(random_seed)
    random.shuffle(tokenized_data)

    # have 100 elements in the test set but a maximum of 25% of the total
    divider_index = min(100, math.floor(len(tokenized_data) / 4))

    test = tokenized_data[:divider_index]
    train = tokenized_data[divider_index:]

    return DataSplit(train, test)

In [36]:
# load, tokenize and split the Tweets
trump_tweets_dataset = split_and_tokenize(load_trump_tweets("data/tweets_01-08-2021.json"))
obama_tweets_dataset = split_and_tokenize(load_obama_tweets("data/Tweets-BarackObama.csv"))
biden_tweets_dataset = split_and_tokenize(load_biden_tweets("data/JoeBidenTweets.csv"))

print("Trump tweets: {} train and {} test".format(len(trump_tweets_dataset.train), len(trump_tweets_dataset.test)))
print("Obama tweets: {} train and {} test".format(len(obama_tweets_dataset.train), len(obama_tweets_dataset.test)))
print("Biden tweets: {} train and {} test".format(len(biden_tweets_dataset.train), len(biden_tweets_dataset.test)))

Trump tweets: 46594 train and 100 test
Obama tweets: 6751 train and 100 test
Biden tweets: 5964 train and 100 test


### Train N-gram Models

2.1 Train n-gram models with n = [1, ..., 5] for Obama, Trump, and Biden.

2.2 Also train a joint model, that will serve as background model.

In [37]:
NGrams = Dict[Tuple[str, ...], int]


def get_single_sentence_n_grams(n: int, tokens: TokenList) -> NGrams:
    """
    Calculates the n-grams for the given token list (i.e., a single sentence)
    """
    n_grams: NGrams = {}
    for i in range(len(tokens) - n + 1):
        key = tuple(tokens[i:n+i])
        if key in n_grams:
            n_grams[key] += 1
        else:
            n_grams[key] = 1

    return n_grams


def get_n_grams(n: int, data: DataSet) -> NGrams:
    """Calculates the n-grams for the given n"""
    n_grams: NGrams = {}
    for tokens in data:
        token_n_grams = get_single_sentence_n_grams(n, tokens)
        for key in token_n_grams.keys():
            if key in n_grams:
                n_grams[key] += token_n_grams[key]
            else:
                n_grams[key] = token_n_grams[key]

    return n_grams


def build_n_gram_models(n: int, data: TokenList) -> Dict[int, NGrams]:
    """
    To predict the first few words of the Tweet, we need the smaller n-grams as
    well. This method does calculate all n-grams up to the given n.
    """
    n_grams_dict: Dict[int, NGrams] = {}
    for i in range(min(n, 2), n + 1):
        n_grams_dict[i] = get_n_grams(i, data)

    return n_grams_dict

In [38]:
def calculate_candidates_and_weights(prev: TokenList, n_gram_model: NGrams):
    """
    Gets all next-token candidates and their weights for the given n_grams
    and previous tokens.
    The size of the previous tokens must be exactly one less than the n-value
    of the n-gram.
    """
    candidates: List[str] = []
    weights: List[int] = []
    for key in n_gram_model.keys():
        if key[:-1] == tuple(prev):
            candidates.append(key[-1])
            weights.append(n_gram_model[key])
    return candidates, weights


def get_suggestion(prev: TokenList, n_gram_model: NGrams) -> str:
    """
    Gets the next random word for the given n_grams.
    The size of the previous tokens must be exactly one less than the n-value
    of the n-gram, or it will not be able to make a prediction.
    """
    candidates, weights = calculate_candidates_and_weights(prev, n_gram_model)
    choices = random.choices(candidates, weights)
    return choices[0]


def get_random_tweet(n: int, n_gram_models: Dict[int, NGrams]) -> str:
    """Generates a random tweet using the given data set."""
    words: List[str] = ["<s>"]
    while words[-1] != "</s>":
        prev = words[-(n-1):] if n > 1 else []
        n_grams = n_gram_models[len(prev) + 1]
        words += [get_suggestion(prev, n_grams)]

    return " ".join(words)

In [39]:
# generate some random Tweets
n_gram_models = build_n_gram_models(n=5, data=trump_tweets_dataset.train)
for _ in range(5):
    random_tweet_trump = get_random_tweet(n=4, n_gram_models=n_gram_models)
    print(random_tweet_trump)

<s> The Democrats want to impeach me . Don ’ t listen to people that hate the U . S . Energy Industry — no fracking , no energy , and respond as you have been a pleasure to host my friend @JPN_PMO @AbeShinzo and his delegation at Mar-a-Lago for our great citizens expect their finances to improve next year , the Nov 3rd Election result may NEVER BE ACCURATELY DETERMINED , which is so terrible , especially since he has not lifted a finger for USMC Tahmooressi . He only lasted 11 days ! https://t.co/RzX3zjXzga </s>
<s> The Democrats are overplaying their hand . They lost their way and are no longer needed at the White House than a politician . If I stayed in Endless Wars forever , they would have had to pay fair taxes , its stock would crash and it would be if @SnoopDogg , failing career and all , for working so long and hard " " </s>
<s> Congratulations to @PiersMorgan on winning @BritishGQ TV Personality Of The Year . Piers deserves his success ! </s>
<s> " " @jjprl : Just left @TrumpDo

### Classify the Tweets

3.1 Use the log-ratio method to classify the Tweets for Trump vs. Biden. Trump should be easy to spot; but what about Obama vs. Biden?

3.2 Analyze: At what context length (n) does the system perform best?

In [40]:
def get_token_probability(
    prev: TokenList,
    token: str,
    n_grams: NGrams
) -> float:
    """
    Gets the probability for the given token.
    The size of the previous tokens must be exactly one less than the n-value
    of the n-gram, or it will not be able to make a prediction.
    """
    candidates, weights = calculate_candidates_and_weights(prev, n_grams)

    if (len(candidates) == 0):
        return 0

    token_weight = \
        weights[candidates.index(token)] if token in candidates else 0

    return token_weight / sum(weights)


def calculate_single_token_log_ratio(    
    prev: TokenList, 
    token: str, 
    n_gram_model1: NGrams, 
    n_gram_model2: NGrams
) -> float:
    """Calculates the log ration of a token for two different n-grams"""
    prob_1 = get_token_probability(prev, token, n_gram_model1)
    prob_2 = get_token_probability(prev, token, n_gram_model2)

    # prevent division by or log of zero if one of the probabilities is 0
    if prob_1 == 0 and prob_2 == 0:
        return 0
    elif prob_1 == 0:
        return -5
    elif prob_2 == 0:
        return 5

    return math.log(prob_1 / prob_2)


def classify(
    n: int, 
    tokens: TokenList, 
    n_gram_models1: Dict[int, NGrams], 
    n_gram_models2: Dict[int, NGrams]
):
    """
    Checks which of the two given datasets is more likely for the given Tweet.
    If true is returned, the first one is more likely, otherwise the second.
    """
    log_ratios = []

    for i in range(1, len(tokens)):
        prev = tokens[:i][-(n-1):] if n > 1 else []
        token = tokens[i]
        n_grams1 = n_gram_models1[len(prev) + 1]
        n_grams2 = n_gram_models2[len(prev) + 1]

        log_ratio = \
            calculate_single_token_log_ratio(prev, token, n_grams1, n_grams2)

        log_ratios.append(log_ratio)

    return sum(log_ratios) > 0


In [41]:
def validate(n, data1, data2, classify_fn):
    """
    Trains the n-gram models on the train data and validates on the test data.
    Uses the implemented classification function to predict the Tweeter.
    """
    n_gram_models1 = build_n_gram_models(n, data1.train)
    n_gram_models2 = build_n_gram_models(n, data2.train)

    true_count = 0
    false_count = 0

    for test_tokens in data1.test:
        result = classify_fn(n, test_tokens, n_gram_models1, n_gram_models2)
        if result:
            true_count += 1
        else:
            false_count += 1

    print("Correct guesses:", true_count, "of", true_count + false_count)

In [42]:
# classify with log-ratio
for i in range(1, 5):
    print("n = {}".format(i))
    n = i
    validate(n, trump_tweets_dataset, biden_tweets_dataset, classify_fn=classify)
    validate(n, obama_tweets_dataset, biden_tweets_dataset, classify_fn=classify)
    print()

n = 1
Correct guesses: 98 of 100
Correct guesses: 86 of 100

n = 2
Correct guesses: 96 of 100
Correct guesses: 88 of 100

n = 3
Correct guesses: 92 of 100
Correct guesses: 81 of 100

n = 4
Correct guesses: 91 of 100
Correct guesses: 81 of 100



### Compute Perplexities

4.1 Compute (and plot) the perplexities for each of the test tweets and models. Is picking the Model with minimum perplexity a better classifier than in 3.1?

In [47]:
def get_whole_sentence_probability(
    n: int,
    tokens: TokenList,
    n_gram_models: Dict[int, NGrams],
) -> float:
    """
    Gets the log probability, that our generator generated the given sentence
    (i.e., token list).
    """
    sentence_probability = 0.

    for i in range(1, len(tokens)):
        prev = tokens[:i][-(n-1):] if n > 1 else []
        token = tokens[i]
        n_grams = n_gram_models[len(prev) + 1]

        token_probability = get_token_probability(prev, token, n_grams)

        # avoid zero
        token_probability = max(token_probability, 0.01)

        sentence_probability = sentence_probability + np.log(token_probability)

    return sentence_probability / len(tokens)


def calculate_perplexity(
    n: int,
    tokens: TokenList,
    n_gram_models: Dict[int, NGrams],
) -> float:
    """Calculates the preplexity for the given tokens."""
    sentence_probability = \
        get_whole_sentence_probability(n, tokens, n_gram_models)

    if sentence_probability == 0:
        return math.inf

    return np.exp(-1 * sentence_probability)


def classify_with_perplexity(
    n: int, 
    tokens: TokenList, 
    n_gram_models1: Dict[int, NGrams], 
    n_gram_models2: Dict[int, NGrams]
):
    """
    Checks which of the two given datasets is more likely for the given Tweet.
    If true is returned, the first one is more likely, otherwise the second.
    """
    perplexity1 = calculate_perplexity(n, tokens, n_gram_models1)
    perplexity2 = calculate_perplexity(n, tokens, n_gram_models2)

    return perplexity1 < perplexity2

In [48]:
# classify with perplexity
for i in range(1, 5):
    print("n = {}".format(i))
    n = i
    validate(n, trump_tweets_dataset, biden_tweets_dataset, classify_fn=classify_with_perplexity)
    validate(n, obama_tweets_dataset, biden_tweets_dataset, classify_fn=classify_with_perplexity)
    print()

n = 1
Correct guesses: 69 of 100
Correct guesses: 91 of 100

n = 2
Correct guesses: 94 of 100
Correct guesses: 87 of 100

n = 3
Correct guesses: 93 of 100
Correct guesses: 80 of 100

n = 4
Correct guesses: 88 of 100
Correct guesses: 74 of 100

